# BiRefNet Fine-tune (Colab Pro A100) — Faz 3

Bu notebook, `my-bg-remover` projesinin **Faz 3** (fine-tune) adımını Google
Colab'da (A100, 40GB) uçtan uca çalıştırır: `ZhengPeng7/BiRefNet_HR-matting`
(MIT lisanslı) başlangıç ağırlıklarından, Faz 2'de `MyDrive/bg-remover-data/`
altına materyalize edilmiş ~28k çiftlik BiRefNet-formatı veri setiyle (`TRAIN/im/*.jpg`
+ `TRAIN/gt/*.png`, artı `stats.json` + `train_composites_manifest.jsonl`)
fine-tune eder.

**Neden BiRefNet_HR-matting (RMBG-2.0 değil)?** Lisans kararı (2026-07-09,
`docs/superpowers/specs/2026-07-09-bg-remover-design.md`): bitmiş model MIT ile
açık kaynak paylaşılacak. RMBG-2.0'dan fine-tune, CC BY-NC türetilmiş eser
riskini miras alırdı. BiRefNet_HR-matting MIT lisanslı — hem lisans uyumu hem
de kamuflaj kategorisinde ölçülen üstünlüğü (bkz. `docs/reports/2026-07-faz2-veri.md`
§3: MAE'de rmbg-2.0'a göre ~%47 daha iyi) nedeniyle başlangıç noktası seçildi.

## Bu notebook'un dayandığı ARAŞTIRILMIŞ kaynaklar (uydurma config anahtarı YOK)

Aşağıdaki tüm dosyalar bu notebook yazılırken `curl` ile gerçek GitHub/HuggingFace
içeriğinden çekilip **okundu** (2026-07-09, `ZhengPeng7/BiRefNet` `main` dalı):

- `config.py` — `Config.__init__`: görev (`task`) seçimi, `training_set` otomatik
  keşfi (`data_root_dir/{task}/*` altındaki, `testsets` dışındaki alt klasörler),
  `size=(1024,1024)` (Matting görevi için varsayılan, `General-2K` hariç tüm
  görevlerde), `lambdas_pix_last` (görev bazlı loss ağırlıkları), `finetune_last_epochs`
  (Matting: **-10**), `lr` formülü (`(1e-4 if 'DIS5K' in task else 1e-5) * sqrt(batch_size/4)`),
  `mixed_precision=['no','fp16','bf16','fp8'][2]` (**bf16 varsayılan**), `compile=True`.
- `train.py` — `Trainer._train_batch` / `train_epoch`: kayıp hesaplama sırası
  (pix loss + cls loss + gdt/out_ref loss), `finetune_last_epochs` eşiği
  (`epoch > args.epochs + config.finetune_last_epochs`), checkpoint kaydetme
  koşulu (`epoch >= args.epochs - config.save_last and epoch % config.save_step == 0`
  — yalnız SON birkaç epoch'ta, `train.sh`'deki `val_last`/`step` değerlerine göre).
- `dataset.py` — `MyData`: `<data_root_dir>/<task>/<dataset>/{im,gt}` dizin
  sözleşmesi, `transform_image`/`transform_label` (ImageNet normalize), eğitimde
  `(image, label, class_label)` üçlüsü döndürür.
- `models/birefnet.py` — `BiRefNet.forward`: **`self.training` bayrağına göre**
  farklı çıktı yapısı döndürür (`[scaled_preds, class_preds_lst]` eğitimde,
  yalnız `scaled_preds` eval'de — bkz. `bgr/segmenter.py`'nin zaten kullandığı
  `model(inp)[-1].sigmoid()` deseni, bu notebook'ta da AYNEN kullanılıyor).
- `loss.py` — `PixLoss`/`ClsLoss`: hangi loss terimlerinin AKTİF olduğu
  (`lambdas_pix_last[k]` sıfırsa o terim hiç hesaplanmaz).
- `utils.py` — `check_state_dict` (`module.`/`_orig_mod.` önek temizleme),
  `set_seed`.
- README (`Fine-tuning on Custom Data` bölümü) — resmi rehber: veriyi
  `${data_root_dir}/{TASK}/{DATASET}/{im,gt}` altına koy, `--resume` ile devam
  et (epoch numarası dosya adından — `epoch_N.pth` — devam eder), tek-GPU'da
  (A100-40G) doğrudan eğitilebilir olduğu doğrulanmış (yazarın kendi ifadesiyle).
- HuggingFace model kartı (`ZhengPeng7/BiRefNet_HR-matting`) — **önemli bulgu:**
  bu checkpoint aslında **2048×2048**'de eğitilmiş (`config.json`'da `bb_pretrained: false`,
  kart açıklaması "trained with images in 2048x2048"), `BiRefNet-matting` (HR
  olmayan kardeşi) ise 1024×1024. Yükleme yöntemi: `git clone` + `from models.birefnet
  import BiRefNet; BiRefNet.from_pretrained('ZhengPeng7/BiRefNet_HR-matting')`
  (kod GitHub'dan güncel, ağırlık HuggingFace'ten — kartın kendi önerdiği yöntem).

## BİLİNÇLİ SAPMALAR (resmi `train.py`'yi olduğu gibi çalıştırmak YERİNE neden özel bir döngü?)

Resmi `train.py` + `train.sh`'nin gerçek davranışı, bu görevin gereksinimleriyle
üç noktada ÇATIŞIYOR — bu yüzden resmi modülleri (`config.py`, `dataset.py`,
`models/birefnet.py`, `loss.py`, `utils.py`) OLDUĞU GİBİ import edip kullanıyoruz,
ama `train.py`'nin döngüsünü DEĞİL, kendi ince döngümüzü yazıyoruz:

1. **Checkpoint kaydetme çok seyrek:** `train.py`, yalnız `epoch >= args.epochs -
   config.save_last` (Matting: son **50** epoch) VE `epoch % config.save_step == 0`
   (Matting: her **5** epoch'ta bir) checkpoint kaydeder — 100 epoch'luk bir
   koşuda epoch 1-50 arası HİÇ checkpoint yok. Colab oturumu bu pencerede
   koparsa TÜM ilerleme kaybolur. Görev gereksinimi ("her epoch Drive'a
   checkpoint") bunu gerektiriyor — HER epoch kaydediyoruz.
2. **Gradient accumulation resmi kodda YOK:** `train.py`, `Accelerator(...,
   gradient_accumulation_steps=1, ...)` ile SABİT kodlanmış; `accelerator.accumulate(
   self.model)` context'i dosyada YORUM SATIRINDA bırakılmış, kullanılmıyor.
   A100-40G'de bs=2 fiziksel + gradyan biriktirme gerektiği için (bkz. tasarım notu,
   BiRefNet issue #140: bs=1'de bilinen checkpoint bug'ı) bunu kendimiz ekliyoruz
   (`ACCUM` parametresi, manuel `loss/ACCUM` ölçekleme + periyodik `optimizer.step()`).
3. **Checkpoint yalnız model ağırlığı, optimizer/lr_scheduler state'i YOK:**
   `train.py`'nin `torch.save(...)` çağrısı yalnız `state_dict()` kaydeder.
   Çok-oturumlu (birkaç saatte bir kopan) bir Colab eğitiminde momentum/lr
   programının her seferinde sıfırlanması yakınsamayı ciddi bozar — bu yüzden
   checkpoint'imize `optimizer`/`lr_scheduler`/`epoch`'u da ekliyoruz.

Buna karşın, kayıp fonksiyonu sırası, `finetune_last_epochs` eşiği, `check_state_dict`
ile önek temizleme gibi TÜM diğer mantık resmi kod ile BİREBİR aynı tutuluyor
(satır referansları yukarıda) — yalnız üç maddedeki somut çatışma noktaları
değiştirildi, hepsi bu hücrelerde AÇIKÇA yorumlanmış durumda.

**Bu notebook ÇALIŞTIRILMADAN yazılmıştır** (geliştirme ortamında Colab/GPU/Drive
yok) — `training/prepare_data_colab.ipynb` ile aynı yöntem: her hücre bağımsız
+ açıklamalı, yalnız statik doğrulama (JSON/nbformat + her hücrenin `ast.parse`'ı,
artı sampler/resume mantığının GPU'suz simülasyonu — bkz. `tests/test_train_colab_lib.py`)
yapıldı. İlk gerçek Colab koşusunda çıkabilecek sürprizler (ör. `torch.compile`
sürüm uyumsuzluğu, gerçek bellek tavanı) için hücrelerde `try/except` + net hata
mesajları bırakıldı.

## Parametreler

Aşağıdaki değerleri ihtiyacınıza göre değiştirin. **ML bilmiyorsanız** her
parametrenin yanındaki açıklamayı okuyun — çoğu için varsayılan değer zaten
makul bir seçim.

**v3 notu:** RESUME otomatik — Drive'daki `epoch_2.pth`'den (v2) devam eder,
`EPOCHS=4` ile 3. ve 4. epoch'lar eğitilir. v3'ün gerekçesi iki gerçek-veri
bulgusuna dayanıyor (bkz. `.superpowers/sdd/v3-hazirlik-report.md`): (1)
**domain gap** — gerçek-fotoğraf benchmark'ında over-deletion'ın kalıcı
olmasının nedeni camouflage DIŞINDAKİ tüm kategorilerin yalnız SENTETİK
kompozit arka planlarda eğitilmesiydi; `scripts/make_composites.py` artık her
kategoriye orijinal-arka-plan kopyaları (`_o00`) ekliyor. (2) **saydamlık
hedefi** — transparent MAE v1→v2 arası KÖTÜLEŞTİ (0.0437→0.0481, ideogram
hedefi 0.0343); `SAMPLER_PRESET="v3"` transparent payını %20'den %24'e
yükseltir (camouflage'ın v2'de bıraktığı marjdan alınarak). Veri seti ~14k
yeni `_o00` çiftiyle büyüdüğü için `EPOCH_NUM_SAMPLES=27715` ile epoch
uzunluğu v2 ile PARİTEDE (aynı Colab birim maliyeti) sabitlendi.

### MALİYET TABLOSU — `EPOCHS`'u artırmadan önce OKUYUN

~28k çiftlik veri setiyle bir epoch'un maliyeti (A100, 1024px, bs=2 — teorik
hesap; İLK epoch bitince ÖLÇÜLMÜŞ süre + birim tahmini konsola yazdırılır):

| Kalem | Değer |
|---|---|
| İterasyon/epoch | ~13.700 (~27.4k train çifti / bs=2) |
| Süre/epoch | **~1.7–3.4 saat** |
| Colab birimi/epoch | **~20–45 birim** (A100 ≈ 11–13 birim/saat) |
| `EPOCHS=2` toplam | ~3.4–6.8 saat ≈ **40–90 birim** |
| `EPOCHS=100` toplam | ~170–340 saat ≈ **2000–4000 birim** — aylık Colab Pro (~100 birim) ile **KARŞILANAMAZ** |

Bu yüzden v2 varsayılanı `EPOCHS=2`: kısa, ödenebilir bir devam koşusu —
`RESUME="auto"` ile Drive'daki `epoch_1.pth`'den (v1) başlanır, yeni sampler
dengesiyle (`SAMPLER_PRESET="v2"`) 2 epoch daha eğitilir, sonra 155'lik LOKAL
benchmark'la kalite ölçülür; işe yarıyorsa ek birim alınıp yine
`RESUME="auto"` ile kaldığı yerden devam edilir (checkpoint/resume mimarisi
tam da bu kademeli senaryo için kuruldu). Daha uzun bir koşuya ancak maliyeti
bilerek karar verin.

Not: `EPOCHS ≤ 10` iken BiRefNet'in "son 10 epoch loss-reweight" fine-tune
hilesi OTOMATİK atlanır (`train_colab_lib.should_apply_finetune_reweight` —
kısa koşuda pencere epoch 1'in öncesine taşar ve decay üssü daha ilk epoch'ta
n>1 olurdu; ayrıntı fonksiyon docstring'inde).


In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # ÖLÇÜLMÜŞ OOM düzeltmesi (v2) —
                                                                      # CUDA bellek ayırıcısını genişleyebilir
                                                                      # segmentlerle çalıştırıp parçalanmayı
                                                                      # azaltır; A100-40GB'de bs=2 uzun
                                                                      # koşularda gözlemlenen OOM'u giderdi
                                                                      # (bkz. v2-hazırlık raporu). torch import
                                                                      # edilmeden ÖNCE ayarlanmalı — bu yüzden
                                                                      # dosyanın en başında.

# --- Repo kaynağı: ikisinden YALNIZ BİRİNİ doldurun (prepare_data_colab.ipynb
#     ile AYNI kural — projeyi GitHub'a pushladıysanız URL, yoksa Drive'a
#     yüklediğiniz bir zip) ---
REPO_GIT_URL = ""  # ör. "https://github.com/<kullanici>/my-bg-remover.git"
REPO_ZIP_ON_DRIVE = ""  # ör. "/content/drive/MyDrive/my-bg-remover.zip"

# --- Eğitim parametreleri (ML bilmeyenler için özet) ---
EPOCHS = 4            # v3: RESUME otomatik Drive'daki epoch_2.pth'den (v2) devam eder, bu oturum
                      # 3. ve 4. epoch'ları eğitir. Modelin veriyi kaç kez baştan sona göreceği.
                      # DİKKAT: her epoch ~1.7-3.4 saat / ~20-45 Colab birimi tutar (yukarıdaki
                      # MALİYET TABLOSU'nu okuyun) — EPOCH_NUM_SAMPLES sabitlendiği için (aşağı
                      # bakın) veri seti büyüse de epoch maliyeti v2 ile PARİTEDE kalır.
BATCH = 2             # v2: ÖLÇÜLMÜŞ, ÇALIŞAN konfigürasyon. Her adımda GPU'ya aynı anda kaç
                      # görsel verileceği. A100-40GB'de 1024px + Swin-L backbone için bs=2
                      # fiziksel limit civarı (bs=1'de bilinen bir checkpoint hatası var —
                      # BiRefNet issue #140, bkz. tasarım dokümanı — bu yüzden 1 KULLANMAYIN;
                      # aşağıdaki `assert BATCH >= 2` NEDEN yorum satırına alındığına dair NOT'a
                      # bakın).
ACCUM = 4             # v2: ÖLÇÜLMÜŞ, ÇALIŞAN konfigürasyon. "Gradient accumulation": GPU belleği
                      # bs=8 gibi büyük bir grubu aynı anda kaldıramadığı için, ACCUM tane
                      # bs=2'lik küçük grubu arka arkaya işleyip kayıplarını TOPLAYIP tek
                      # seferde güncelleme yapıyoruz — matematiksel olarak bs=(BATCH*ACCUM)=8
                      # ile eğitim yapmışız gibi bir etki (repo'nun kendi referans batch'i 8'dir,
                      # bkz. config.py `batch_size=8` yorumu).
LR = None             # Öğrenme oranı. None = repo'nun resmi formülü otomatik hesaplanır (bkz.
                      # train_colab_lib.effective_lr — efektif batch'e göre ölçeklenir), SONRA
                      # LR_SCALE ile çarpılır (aşağıya bakın). ML bilmiyorsanız DOKUNMAYIN, None
                      # bırakın.
                      # NOT: RESUME ile devam edilen oturumlarda bu parametre ETKİSİZDİR —
                      # checkpoint'teki optimizer durumu (içindeki eski lr dahil) geri
                      # yüklenir; lr'yi oturum ortasında değiştirmek isterseniz optimizer
                      # param_groups'u elle güncellemeniz gerekir.
LR_SCALE = 0.5        # v2: resmi formülden (`train_colab_lib.effective_lr`) hesaplanan lr'yi
                      # ÇARPAR — yalnız LR=None iken (yani lr formülle otomatik hesaplanıyorken)
                      # uygulanır; LR elle set edilmişse (LR != None) LR_SCALE UYGULANMAZ, elle
                      # girilen değer olduğu gibi kullanılır. Varsayılan 0.5: v1'in forgetting'ini
                      # azaltmak için daha düşük bir lr ile devam ediyoruz — devam/fine-tune
                      # koşularında lr'yi küçültmek unutmayı azaltır (literatür standardı).
                      # 1.0 = eski (v1) davranış, formül hiç ölçeklenmez.
RESUME = "auto"        # "auto" = Drive'da (yoksa yerel diskte) en son checkpoint'i OTOMATİK
                      # bulup oradan devam eder — v2'de bu Drive'daki epoch_1.pth'dir: RESUME
                      # otomatik, epoch_1.pth'den devam eder, kazanımlar korunur. Hiç checkpoint
                      # yoksa BiRefNet_HR-matting'ten SIFIRDAN başlar. Colab oturumu kopsa bile
                      # bu notebook'u yeniden çalıştırmak yeterli.
DATA_DIR = "bg-remover-data"          # Drive'daki veri klasörü adı (Faz 2 notebook'unun çıktısı).
N_EVAL_EVERY = 2      # Her kaç epoch'ta bir, küçük bir doğrulama örneklemi (varsayılan
                      # 24 görsel) üzerinde hızlı bir kalite ölçümü (MAE) yapılıp
                      # train_log.txt'e yazılsın. EPOCHS=4'lük v3 devam koşusunda 2 ölçüm
                      # noktası anlamına gelir (3. ve 4. epoch sonları).
EPOCH_NUM_SAMPLES = 27715        # v3: WeightedRandomSampler'ın epoch başına çekeceği SABİT örnek
                                  # sayısı (bkz. train_colab_lib.resolve_sampler_num_samples) — v2'nin
                                  # epoch büyüklüğüyle PARİTE (~27.4k train çifti / bs=2, yukarıdaki
                                  # MALİYET TABLOSU). `_o00` kopyalarıyla veri seti ~14k büyüse bile
                                  # epoch maliyeti (~48 birim) sabit kalır — None bırakılırsa (v1/v2
                                  # davranışı) eski gibi len(train_dataset) kullanılır.

# --- İleri düzey / nadiren değiştirilecek parametreler ---
SEED = 42                      # Faz 2 ile TUTARLI aynı tohum (VAL bölünmesi + reprodüksiyon).
VAL_FRACTION = 0.02             # Eğitim setinden AYRILIP hiç eğitimde kullanılmayacak pay (%2).
QUICK_EVAL_N = 24                # VAL kümesinden, HER hızlı-değerlendirmede kullanılacak SABİT görsel sayısı.
SAMPLER_PRESET = "v4"            # "v4" (varsayılan) = train_colab_lib.SAMPLER_PRESET_V4 (camouflage %12,
                                  # transparent %18, hair %8, complex %19, thin %13, general %4, text %10,
                                  # fx %8, illustration %8) — v3'ün GERÇEK benchmark sonuçlarına göre
                                  # ayarlandı: odak complex+thin'de ve YENİ yeteneklerde (logo/yazı
                                  # koruma=text, obje etrafı VFX parıltı=fx, illüstrasyon=illustration —
                                  # verileri v4_veri_guncelleme_hucresi.py üretir). transparent %18'de
                                  # TUTULDU (Ideogram'ın 0.0343 hedefine 0.0043 kaldı, kovalamaca sürüyor);
                                  # camouflage %12'ye DÜŞTÜ (v3 0.0304 vs Ideogram 0.1179 — marj devasa);
                                  # hair %8 (0.0067 MAE, rmbg'nin 0.0045'ine yakın — pay azaltılabilir).
                                  # "v3"/"v2"/"v1" eski davranışlar için hâlâ seçilebilir — bkz. hücre (e),
                                  # tcl.SAMPLER_PRESETS[SAMPLER_PRESET] ile çözümlenir.
KEEP_LAST_N_CHECKPOINTS = 3      # Hem yerelde hem Drive'da yalnız en son N epoch'un checkpoint'i tutulur (disk kotası;
                                  # 3 paket ≈ 8-9GB Drive alanı — bkz. eğitim döngüsü hücresinin Drive kotası notu).
NUM_WORKERS = 6                  # DataLoader'ın paralel veri-okuma işçi sayısı. Resmi formül
                                  # (min(num_workers, batch_size)) bs=2'de yalnız 2 işçiye izin verirdi —
                                  # 13.7k iterasyonluk epoch'ta veri okuma darboğaz olur; A100 VM'lerinin
                                  # bol CPU çekirdeği varken bunu batch'ten AYRIŞTIRIYORUZ (bilinçli sapma).
COMPILE_MODEL = False            # torch.compile ~%40 hızlandırabilir (config.py yorumu) ama derleme TEMBELDİR:
                                  # buradaki try/except İLK BATCH'e kadar hata yakalayamaz. İlk koşuda False
                                  # (güvenli taban); True yaparsanız torch._dynamo.config.suppress_errors=True
                                  # da açılır (derlenemeyen graf parçaları sessizce eager'a düşer) — bkz.
                                  # model kurulum hücresinin notu.
HF_MODEL_ID = "ZhengPeng7/BiRefNet_HR-matting"   # Başlangıç ağırlıkları (MIT) — RMBG-2.0 DEĞİL (lisans kararı).
BIREFNET_GIT_URL = "https://github.com/ZhengPeng7/BiRefNet.git"

DRIVE_ROOT = "/content/drive/MyDrive"
DRIVE_CKPT_SUBDIR = "bg-remover-checkpoints"     # her epoch buraya kopyalanır (görev gereksinimi).
DRIVE_STATUS_SUBDIR = "bg-remover-status"        # train_log.txt buraya yazılır (kontrolcü bunu dışarıdan izler).
WORKDIR = "/content/my-bg-remover"               # bizim repo'muzun klonlanacağı yer (train_colab_lib.py + benchmark.metrics için).
BIREFNET_DIR = "/content/BiRefNet"               # BiRefNet resmi repo'sunun klonlanacağı yer.
LOCAL_DATA_ROOT = "/content/dis_data"            # Drive verisinin KOPYALANACAĞI yerel disk (I/O hızı için — Drive-mount üzerinden
                                                  # okuma çok yavaş, bkz. tasarım dokümanı §3 "Colab oturumu kopması" notu).
BIREFNET_TASK = "Matting"       # config.py'nin 6 hazır görevinden biri (DIS5K/COD/HRSOD/General/General-2K/Matting).
                                 # "Matting" seçildi çünkü: (1) başlangıç ağırlığımız (HR-matting) bu görevle eğitilmiş,
                                 # (2) loss seti (bce+mae+ssim, iou YOK) yumuşak-alfa (transparent/hair) + sert-kenar
                                 # (camouflage/general) karışımını dengeli işliyor, (3) size=(1024,1024) zaten bu görevin
                                 # varsayılanı — testset çözünürlük p50'siyle (854px, bkz. stats.json) uyumlu.

assert REPO_GIT_URL or REPO_ZIP_ON_DRIVE, "REPO_GIT_URL veya REPO_ZIP_ON_DRIVE alanlarından biri dolu olmalı"
# assert BATCH >= 2, "bs=1'de bilinen bir checkpoint hatası var (BiRefNet issue #140) — BATCH>=2 kullanın"
# NOT (v2): yukarıdaki assert KALDIRILDI (yorum satırına alındı, silinmedi). PYTORCH_CUDA_ALLOC_CONF
# (bu hücrenin başında) ölçülmüş bir OOM düzeltmesi olarak eklendi ama bazı VM'lerde/oturumlarda
# BATCH=2 hâlâ marjinal kalabilir; kullanıcı bilinçli olarak BATCH=1 deneyebilsin diye zorunlu
# kısıtlama kaldırıldı (issue #140'ın checkpoint hatası riski HÂLÂ GEÇERLİ — yalnız artık kullanıcı
# tercihi, notebook tarafından ZORUNLU kılınmıyor). Varsayılan yine BATCH=2 (ölçülmüş, çalışan
# konfigürasyon) — bu satırı yalnız OOM ısrar ederse ve BATCH=1'i bilerek denemek isterseniz göz
# önünde bulundurun.
print("Parametreler ayarlandı.")


## (a) Drive'ı Bağla + Repoları Getir

İki repo klonlanıyor: **BiRefNet** (resmi eğitim kodu — `config.py`, `train.py`,
`dataset.py`, `models/`, `loss.py`, `utils.py`) ve **kendi projemiz**
(`my-bg-remover` — yalnız `training/train_colab_lib.py` — sampler/resume/checkpoint
budama saf-Python mantığı, tek doğruluk kaynağı olsun diye kopyalanmıyor tekrar
yazılmıyor — ve `benchmark/metrics.py` — hızlı-değerlendirme MAE hesabı için,
proje genelinde kullanılan AYNI fonksiyon).

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

os.makedirs(f"{DRIVE_ROOT}/{DRIVE_CKPT_SUBDIR}", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/{DRIVE_STATUS_SUBDIR}", exist_ok=True)
print("Drive bağlandı.")

In [ ]:
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path


def _clone_or_extract(repo_git_url: str, repo_zip_on_drive: str, workdir: str) -> None:
    if Path(workdir).exists():
        print(f"{workdir} zaten mevcut — klonlama/açma atlanıyor (idempotent).")
        return
    if repo_git_url:
        print(f"git clone {repo_git_url} -> {workdir}")
        subprocess.run(["git", "clone", repo_git_url, workdir], check=True)
    else:
        print(f"zip açılıyor: {repo_zip_on_drive} -> {workdir}")
        extract_tmp = f"{workdir}_zip_extract_tmp"
        with zipfile.ZipFile(repo_zip_on_drive) as zf:
            zf.extractall(extract_tmp)
        extracted = list(Path(extract_tmp).iterdir())
        src_root = extracted[0] if len(extracted) == 1 and extracted[0].is_dir() else Path(extract_tmp)
        shutil.move(str(src_root), workdir)


# Kendi projemiz: yalnız train_colab_lib.py (sampler/resume mantığı) + benchmark/
# paketi (MAE) için gerekli — pip install -e . ile "benchmark" paketini kullanılabilir kılıyoruz.
_clone_or_extract(REPO_GIT_URL, REPO_ZIP_ON_DRIVE, WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)
os.chdir(WORKDIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-q"], check=True)
import training.train_colab_lib as tcl  # noqa: E402  (yukarıdaki sys.path eklemesi + namespace-package importu)
import benchmark.metrics as bm  # noqa: E402

# BiRefNet resmi repo'su.
_clone_or_extract(BIREFNET_GIT_URL, "", BIREFNET_DIR)

# DİKKAT: BiRefNet'in requirements.txt'i (GitHub main dalı, doğrulandı) `torch>=2.5.0`,
# `torchvision` ve `numpy<2` satırları içeriyor. Colab'ın önceden kurulu torch'u
# (imajın kendi CUDA sürümüyle EŞLEŞTİRİLMİŞ derleme) ve numpy 2.x'i buna göre
# yeniden kurmaya kalkışmak iki ayrı tuzak: (1) torch için CPU-only tekerlek /
# CUDA sürüm uyuşmazlığı riski, (2) `numpy<2` Colab'ın numpy 2.x'ini DOWNGRADE
# eder — önceden numpy 2 ile derlenmiş yüklü paketler bozulur ve "restart runtime"
# döngüsüne sokar. Bu yüzden torch/torchvision/numpy satırlarını ÇIKARIP kalanını
# kuruyoruz — Colab'ın kendi (zaten uyumlu) kurulumlarına DOKUNMUYORUZ.
req_path = Path(BIREFNET_DIR) / "requirements.txt"
filtered = [
    line for line in req_path.read_text().splitlines()
    if line.strip() and not line.strip().lower().startswith(("torch", "numpy"))
]
filtered_req_path = Path("/content/birefnet_requirements_filtered.txt")
filtered_req_path.write_text("\n".join(filtered) + "\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(filtered_req_path), "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "safetensors", "-q"], check=True)
print("Repolar hazır:", WORKDIR, BIREFNET_DIR)
print(f"torch sürümü (Colab'ın kendisi, DEĞİŞTİRİLMEDİ): {__import__('torch').__version__}")
print(f"numpy sürümü (Colab'ın kendisi, DEĞİŞTİRİLMEDİ): {__import__('numpy').__version__}")

## (b) `config.py`'yi Yama (görev + yol + batch)

BiRefNet'in resmi ince-ayar rehberi (`README.md` "Fine-tuning on Custom Data")
`config.py` içindeki birkaç değişkenin ELLE değiştirilmesini öneriyor
(`sys_home_dir`, görev seçimi, `training_set`/`testsets`). Biz **var olan**
`Matting` görevini kullandığımız için yeni bir görev adı İCAT ETMİYORUZ (rehber
genel olarak "yeni bir görev adı seçin" diyor, ama 6 hazır görevden biri zaten
tam ihtiyacımıza uyuyor — bkz. parametre hücresindeki `BIREFNET_TASK` notu) —
yalnız üç satırı metinsel olarak değiştiriyoruz: (1) `self.task` listesinin
seçili indeksini `Matting`'e çeviriyoruz, (2) `sys_home_dir`'i yerel diskimize
(`LOCAL_DATA_ROOT`'a) yöneltiyoruz — böylece `data_root_dir` Drive'ın
YAVAŞ mount noktasına değil, hızlı yerel diske işaret ediyor, (3) `batch_size`'ı
`BATCH` parametresine eşitliyoruz (bu da `config.lr`/`config.num_workers`
hesaplarını doğru tetikler). `self.bb` (backbone seçimi, `swin_v1_l`) hiç
DOKUNULMUYOR — HR-matting ağırlıkları bu backbone ile eğitildi, değiştirilirse
`from_pretrained` yanlış mimariye state_dict yüklemeye çalışır.

Yama mantığı `train_colab_lib.apply_config_patches` içinde (regex tabanlı,
**idempotent** ve yeniden-parametrelenebilir — aynı VM'de bu hücrenin, aynı
YA DA farklı parametre değerleriyle, yeniden çalıştırılması patlamaz; birim
testli, bkz. `tests/test_train_colab_lib.py::test_apply_config_patches_is_idempotent`).
Desen `main` dalında hiç bulunamazsa (repo değişmişse) net bir `ValueError`
fırlatır — sessizce yanlış davranmaz.

In [ ]:
import re

config_py_path = Path(BIREFNET_DIR) / "config.py"
local_home = str(Path(LOCAL_DATA_ROOT))  # data_root_dir = LOCAL_DATA_ROOT/datasets/dis olacak (hücre (c) ile AYNI kök)
patched = tcl.apply_config_patches(
    config_py_path.read_text(), task=BIREFNET_TASK, sys_home_dir=local_home, batch_size=BATCH
)
config_py_path.write_text(patched)
print(f"config.py yamalandı (idempotent): task={BIREFNET_TASK}, sys_home_dir={local_home}, batch_size={BATCH}")

## (c) Veriyi Yerel Diske Kopyala + Deterministik TRAIN/VAL Bölünmesi

Drive-mount üzerinden binlerce küçük dosya okumak ÇOK YAVAŞ (tasarım
dokümanının da belirttiği gibi) — bu yüzden veriyi BİR KEZ yerel diske
(`/content`, Colab'ın geçici ama HIZLI diski) kopyalıyoruz. `config.py`
sözleşmesi gereği hedef yol: `<data_root_dir>/<TASK>/TRAIN/{im,gt}` (BiRefNet'in
`MyData` sınıfı `training_set` adındaki alt klasörleri otomatik keşfeder —
bkz. `config.py` `datasets_all` satırı).

**VAL bölünmesi FİZİKSEL TAŞIMA değil, deterministik + KALICI bir SEÇİM + ayrı
kopyalama**: `train_colab_lib.load_or_create_val_split` İLK koşuda seçilen val
listesini Drive'a (`bg-remover-status/val_stems.json`) yazar; SONRAKİ tüm
koşular bu dosyadan okur. Neden kalıcılık gerekli: Drive'daki veri seti sonradan
BÜYÜYEBİLİR (Faz 2 pipeline'ı idempotent — yeni koşu yeni çiftler ekleyebilir);
salt-deterministik bölünme girdi listesi değişince FARKLI bir val kümesi
üretir ve önceki epoch'larda eğitimde görülmüş görseller val'e sızardı.
**Belgelenmiş tercih:** sonradan eklenen stem'lerin TAMAMI TRAIN'e gider —
val kümesi ilk koşuda ne seçildiyse o kalır (epoch'lar arası karşılaştırılabilir
MAE trendi, oransal val büyütmekten daha değerli; hızlı-değerlendirme zaten
sabit 24'lük bir alt küme kullanıyor).

VAL çiftleri **`Matting/TRAIN`'e KOPYALANMAZ** (eğitim setinden tamamen hariç
tutulur — sızıntı yok), ayrı bir `val_holdout/{im,gt}` klasörüne konur — BİLİNÇLİ
OLARAK `<data_root_dir>/Matting/` ağacının DIŞINDA (aynı ağacın içine "kardeş"
bir klasör olarak koysaydık, `config.py`'nin `datasets_all` otomatik keşfi bunu
da bir eğitim kaynağı sanıp `training_set`'e dahil ederdi — bkz. bir sonraki
hücrenin yorumu). `MyData` bu klasörü HİÇ görmüyor; yalnız bizim periyodik
hızlı-değerlendirmemiz kullanıyor.

İdempotentlik: hedef dosya zaten varsa VE **hem im hem gt** dosya boyutları
kaynakla birebir eşleşiyorsa yeniden KOPYALANMAZ (`train_colab_lib.copy_pairs`,
birim testli) — kesintiye uğramış bir kopyalamada kesik (truncated) kalmış gt
dosyaları da bir sonraki koşuda otomatik onarılır.

In [ ]:
import time

drive_data_dir = Path(DRIVE_ROOT) / DATA_DIR
drive_train_im = drive_data_dir / "TRAIN" / "im"
drive_train_gt = drive_data_dir / "TRAIN" / "gt"
assert drive_train_im.is_dir() and drive_train_gt.is_dir(), (
    f"Drive'da beklenen veri bulunamadı: {drive_train_im} / {drive_train_gt} — "
    f"prepare_data_colab.ipynb (Faz 2) henüz tamamlanmamış olabilir."
)

all_stems = sorted(p.stem for p in drive_train_im.iterdir() if p.is_file())
print(f"Drive'da toplam {len(all_stems)} çift bulundu.")

# KALICI bölünme: ilk koşuda val listesi Drive'a yazılır, sonraki koşular AYNI
# val kümesini kullanır (veri seti büyüse bile — yeni stem'ler TRAIN'e gider).
val_stems_json = Path(DRIVE_ROOT) / DRIVE_STATUS_SUBDIR / "val_stems.json"
train_stems, val_stems = tcl.load_or_create_val_split(
    all_stems, seed=SEED, val_fraction=VAL_FRACTION, persist_path=val_stems_json
)
print(f"Bölünme (seed={SEED}, kalıcı liste: {val_stems_json}): "
      f"TRAIN={len(train_stems)}, VAL_HOLDOUT={len(val_stems)} (ilk-koşu hedef payı: %{VAL_FRACTION * 100:.1f})")

# DİKKAT: VAL_HOLDOUT'u BİLİNÇLİ OLARAK <data_root_dir>/<TASK>/ ağacının DIŞINA
# koyuyoruz (aynı ağacın içine "kardeş" bir klasör olarak koysaydık, config.py'nin
# `datasets_all = '+'.join([ds for ds in os.listdir(data_root_dir/task) ...])`
# otomatik keşfi bunu da bir eğitim kaynağı sanıp `training_set`'e "TRAIN+VAL_HOLDOUT"
# olarak eklerdi — VAL sızıntısı! Bu yüzden VAL_HOLDOUT tamamen ayrı bir dalda.
local_task_root = Path(LOCAL_DATA_ROOT) / "datasets" / "dis" / BIREFNET_TASK
local_train_im = local_task_root / "TRAIN" / "im"
local_train_gt = local_task_root / "TRAIN" / "gt"
local_val_root = Path(LOCAL_DATA_ROOT) / "val_holdout"
local_val_im = local_val_root / "im"
local_val_gt = local_val_root / "gt"
for d in (local_train_im, local_train_gt, local_val_im, local_val_gt):
    d.mkdir(parents=True, exist_ok=True)


# Kopyalama mantığı train_colab_lib.copy_pairs'te (birim testli): hem im hem gt
# boyutu doğrulanır — kesik kalmış gt dosyaları da onarılır.
t0 = time.time()
n_train_copied = tcl.copy_pairs(train_stems, drive_train_im, drive_train_gt, local_train_im, local_train_gt)
n_val_copied = tcl.copy_pairs(val_stems, drive_train_im, drive_train_gt, local_val_im, local_val_gt)
print(f"Kopyalandı: TRAIN +{n_train_copied} (toplam {len(train_stems)}), "
      f"VAL_HOLDOUT +{n_val_copied} (toplam {len(val_stems)}) — {time.time() - t0:.0f} sn")

EVAL_STEMS = tcl.fixed_eval_subset(val_stems, seed=SEED, n=QUICK_EVAL_N)
print(f"Hızlı-değerlendirme için SABİT {len(EVAL_STEMS)} görsel seçildi (her N_EVAL_EVERY epoch'ta AYNI görseller).")

## (d) Model + Checkpoint/Resume Kurulumu

`RESUME="auto"` ise önce **Drive**'daki (`bg-remover-checkpoints/`), yoksa
**yerel** checkpoint dizinindeki en son `epoch_<N>.pth`'i arar
(`train_colab_lib.find_latest_checkpoint` — regex ile dosya adından epoch
numarasını çıkarır, en büyüğünü seçer). Bulunursa oradan (model + optimizer +
lr_scheduler + epoch sayacı) devam edilir; bulunamazsa `BiRefNet.from_pretrained(
HF_MODEL_ID)` ile SIFIRDAN (HR-matting ağırlıklarından) başlanır — bu README'nin
kendi önerdiği "kod GitHub'dan, ağırlık HuggingFace'ten" yöntemi.

In [ ]:
import torch

sys.path.insert(0, BIREFNET_DIR)
os.chdir(BIREFNET_DIR)  # config.py'nin kendi train.sh keşfi + göreli importlar için

from config import Config  # noqa: E402
from models.birefnet import BiRefNet  # noqa: E402
from utils import check_state_dict, set_seed  # noqa: E402

config = Config()
assert config.task == BIREFNET_TASK
# `config.training_set`, `Config.__init__` içinde `data_root_dir/<TASK>/` altındaki
# TÜM alt klasörleri (testsets hariç) otomatik keşfedip '+' ile birleştiriyor
# (bkz. config.py `datasets_all` satırı). Hücre (c)'de VAL_HOLDOUT'u bu ağacın
# DIŞINA koyduk (bkz. o hücrenin yorumu) ama yine de EMİN OLMAK için burada
# `training_set`'i AÇIKÇA "TRAIN"e sabitliyoruz — otomatik keşfe güvenmek yerine.
if config.training_set != "TRAIN":
    print(f"UYARI: otomatik keşif training_set='{config.training_set}' buldu — 'TRAIN'e zorlanıyor.")
config.training_set = "TRAIN"
print(f"config: task={config.task} size={config.size} batch_size={config.batch_size} "
      f"lr(resmi formül, henüz accum'suz)={config.lr:.2e} finetune_last_epochs={config.finetune_last_epochs}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("UYARI: CUDA bulunamadı — Colab'da Runtime > Değiştir > Donanım hızlandırıcı > A100 seçili mi kontrol edin.")
set_seed(config.rand_seed if config.rand_seed else SEED)

local_ckpt_dir = Path("/content/ckpts")
local_ckpt_dir.mkdir(parents=True, exist_ok=True)
drive_ckpt_dir = Path(DRIVE_ROOT) / DRIVE_CKPT_SUBDIR
drive_ckpt_dir.mkdir(parents=True, exist_ok=True)

resume_info = None
if RESUME == "auto":
    resume_info = tcl.find_latest_checkpoint(drive_ckpt_dir) or tcl.find_latest_checkpoint(local_ckpt_dir)
elif RESUME:
    m = re.search(r"epoch_(\d+)", str(RESUME))
    assert m, f"RESUME yolunda 'epoch_<N>' deseni bulunamadı: {RESUME}"
    resume_info = (RESUME, int(m.group(1)))

payload = None
if resume_info:
    ckpt_path_str, resumed_epoch = resume_info
    local_resume_path = local_ckpt_dir / Path(ckpt_path_str).name
    if str(local_resume_path) != ckpt_path_str:
        shutil.copy2(ckpt_path_str, local_resume_path)  # Drive'dan yerel diske (tek dosya, hızlı) — sonraki torch.load yerelden okusun
    print(f"Checkpoint bulundu: {ckpt_path_str} (epoch {resumed_epoch}) -> devam ediliyor.")
    payload = torch.load(local_resume_path, map_location="cpu", weights_only=False)
    assert isinstance(payload, dict) and "model" in payload, (
        f"Checkpoint beklenen pakette değil: {ckpt_path_str}\n"
        f"Bu notebook'un ürettiği epoch_N.pth dosyaları {{'model','optimizer','lr_scheduler','epoch'}} "
        f"anahtarlı bir dict'tir. Elinizdeki dosya muhtemelen HAM bir state_dict (ör. resmi BiRefNet "
        f"release'lerinden inen epoch_244.pth gibi) — bunlar doğrudan RESUME edilemez çünkü optimizer/"
        f"lr_scheduler/epoch bilgisi içermezler. Ham ağırlıkla başlamak isterseniz dosyayı checkpoint "
        f"dizinlerinden kaldırın ve HF_MODEL_ID'yi kullanın (RESUME='auto' + boş dizin = sıfırdan başlar)."
    )
    model = BiRefNet(bb_pretrained=False)
    model.load_state_dict(check_state_dict(payload["model"]))
    epoch_st = payload.get("epoch", resumed_epoch) + 1
else:
    print(f"Checkpoint yok — {HF_MODEL_ID} başlangıç ağırlıklarından SIFIRDAN başlanıyor.")
    model = BiRefNet.from_pretrained(HF_MODEL_ID)
    epoch_st = 1

model = model.to(device)
if config.precisionHigh:
    torch.set_float32_matmul_precision("high")

# torch.compile TEMBEL çalışır: buradaki sarmalayıcı çağrı neredeyse hiç hata
# üretmez, asıl derleme İLK BATCH'te tetiklenir — yani buradaki try/except tek
# başına yanıltıcı bir güvence olurdu. Bu yüzden: (1) varsayılan COMPILE_MODEL=False
# (ilk koşu için güvenli taban), (2) True seçilirse dynamo'nun suppress_errors'u
# açılır — derlenemeyen graf parçaları eğitimi ÇÖKERTMEK yerine sessizce eager
# moda düşer (o parçalar için hızlanma kaybedilir, doğruluk etkilenmez).
if COMPILE_MODEL:
    import torch._dynamo

    torch._dynamo.config.suppress_errors = True
    model = torch.compile(model, mode="default")
    print("torch.compile etkin (suppress_errors=True — derlenemeyen parçalar eager'a düşer).")
else:
    print("torch.compile KAPALI (COMPILE_MODEL=False — ilk koşu için güvenli varsayılan).")

print(f"Başlangıç epoch: {epoch_st} / hedef {EPOCHS}")

## (e) Kategori Ağırlıklı Örnekleme (`WeightedRandomSampler`)

`scripts/make_composites.py`'nin fiziksel çarpanları (transparent×10,
camouflage×2 — bkz. o dosyanın docstring'i ve `docs/reports/2026-07-faz2-veri.md`
§2) tek başına ≥%20 hedefini garantilemiyor (gerçek Colab sayıları
materyalizasyondan SONRA belli oluyor). **En az müdahaleci** çözüm: resmi
`MyData`/`train.py` kodunu HİÇ değiştirmeden, yalnız `DataLoader`'ın
`sampler=`'ını (`shuffle=is_train, sampler=None` yerine) bir
`WeightedRandomSampler` ile değiştirmek — ağırlıklar Faz 2'nin kompozit
manifest'inden (`train_composites_manifest.jsonl`, id→category, Drive'a
`stage7_drive_copy` ile kopyalanmıştı) hesaplanıyor (bkz. `train_colab_lib.
compute_sample_weights` — hedefli kategoriler tam `target_share` payını alır,
diğerleri kendi aralarında ham oranlarını korur).

Fiziksel oversampling'i (çarpanı 13-15x'e çıkarmak, Faz 2 raporunun ilk taslağı)
BİLİNÇLİ OLARAK reddettik: disk/IO maliyeti yüksek VE `general` kategorisinin
nihai boyutu (0-4000 arası, bkz. faz2 raporu §4) önceden bilinmiyor — sampler
ağırlıkları GERÇEK sayılara göre otomatik ayarlanıyor, fiziksel veri yeniden
üretilmesine gerek yok.

**v2:** Hedef paylar artık `SAMPLER_PRESET` parametresiyle (`"v1"`/`"v2"`) seçiliyor — `tcl.SAMPLER_PRESETS[SAMPLER_PRESET]` (bkz. `train_colab_lib.SAMPLER_PRESET_V1`/`SAMPLER_PRESET_V2`). v1'in yalnız transparent+camouflage'a ≥%20 hedefleyip complex/thin/hair'i hedefsiz bırakması (kalan payın büyük kısmının ham hacmi çok daha büyük olan hair'e gitmesi) v1 karşılaştırma raporundaki catastrophic forgetting'in kök nedeniydi; v2 hair/complex/thin'e de AÇIKÇA hedef vererek bunu düzeltir.


In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler

from dataset import MyData  # noqa: E402  (BIREFNET_DIR sys.path'te, hücre (d)'den)

train_dataset = MyData(datasets=config.training_set, data_size=config.size, is_train=True)
print(f"TRAIN veri seti: {len(train_dataset)} örnek.")

manifest_path = drive_data_dir / "train_composites_manifest.jsonl"
assert manifest_path.exists(), f"Kompozit manifest bulunamadı: {manifest_path}"
stem_category = tcl.load_stem_categories(manifest_path)

stems_in_order = [Path(p).stem for p in train_dataset.image_paths]

# --- v3 ön-uçuş koruması (reviewer bulgusu #2): v3 preset'inin TÜM amacı,
# v3_veri_guncelleme_hucresi.py'nin ürettiği _o00 (orijinal arka plan)
# kopyalarının eğitimde görülmesi. Drive verisi henüz güncellenmemişse bu
# PAHALI GPU koşusu v3'ün bütün anlamını sessizce kaçırırdı — erken ve
# gürültülü şekilde durduruyoruz. ---
o00_count = sum(1 for s in stems_in_order if s.endswith("_o00"))
print(f"_o00 (orijinal arka plan) örnek sayısı: {o00_count} / toplam {len(stems_in_order)}")
if SAMPLER_PRESET in ("v3", "v4") and o00_count == 0:
    raise RuntimeError(
        "v3 verisi yok — önce v3_veri_guncelleme_hucresi.py'yi ücretsiz CPU oturumunda çalıştırın "
        f"(Drive'daki bg-remover-data/TRAIN'de hiç _o00 dosyası bulunamadı; SAMPLER_PRESET={SAMPLER_PRESET!r} "
        "bu veri olmadan anlamsız — pahalı GPU oturumunu boşa harcamamak için durduruldu)."
    )

# --- v4 ön-uçuş koruması (v3 guard'ıyla AYNI amaç): v4 preset'inin yeni
# kategorileri (text/fx/illustration — v4_veri_guncelleme_hucresi.py üretir)
# manifest+TRAIN'de yoksa, %26'lık hedef payları boşa düşer ve bu PAHALI GPU
# koşusu v4'ün bütün anlamını sessizce kaçırırdı — erken ve gürültülü şekilde
# durduruyoruz (her yeni kategoriden en az 100 örnek şartı). ---
if SAMPLER_PRESET == "v4":
    _v4_counts = {c: 0 for c in ("text", "fx", "illustration")}
    for _s in stems_in_order:
        _c = stem_category.get(_s)
        if _c in _v4_counts:
            _v4_counts[_c] += 1
    print(f"v4 yeni kategori örnek sayıları (manifest+TRAIN kesişimi): {_v4_counts}")
    _v4_missing = {c: n for c, n in _v4_counts.items() if n < 100}
    if _v4_missing:
        raise RuntimeError(
            "v4 verisi eksik — önce v4_veri_guncelleme_hucresi.py'yi ücretsiz CPU oturumunda çalıştırın "
            f"(text/fx/illustration kategorilerinin HER BİRİNDEN en az 100 örnek gerekli; eksik: {_v4_missing}; "
            "SAMPLER_PRESET='v4' bu veri olmadan anlamsız — pahalı GPU oturumunu boşa harcamamak için durduruldu)."
        )

n_unknown = sum(1 for s in stems_in_order if s not in stem_category)
if n_unknown:
    print(f"UYARI: {n_unknown}/{len(stems_in_order)} stem manifest'te bulunamadı "
          f"(kategori '_other' varsayılacak — sampler yine de çalışır, yalnız bu "
          f"örnekler hedefli kategori payı hesabına dahil edilmez).")

assert SAMPLER_PRESET in tcl.SAMPLER_PRESETS, (
    f"bilinmeyen SAMPLER_PRESET={SAMPLER_PRESET!r} — geçerli: {sorted(tcl.SAMPLER_PRESETS)}"
)
CATEGORY_TARGET_SHARE = tcl.SAMPLER_PRESETS[SAMPLER_PRESET]
print(f"SAMPLER_PRESET={SAMPLER_PRESET!r} -> hedef paylar: {CATEGORY_TARGET_SHARE}")

sample_weights = tcl.compute_sample_weights(stems_in_order, stem_category, CATEGORY_TARGET_SHARE)
achieved_shares = tcl.compute_expected_shares(sample_weights, stems_in_order, stem_category)
print("Sampler kurulduktan sonra BEKLENEN epoch-içi kategori payları:")
for cat, share in sorted(achieved_shares.items(), key=lambda kv: -kv[1]):
    print(f"  {cat}: %{share * 100:.1f}")

In [ ]:
# Tolerans: SAMPLER_PRESET toplamı kasıtlı olarak <1.0 (kalan pay, manifest'te kategorisi
# bulunamayan "_other" stem'lere ayrılıyor — bkz. train_colab_lib.SAMPLER_PRESET_V2 docstring'i).
# Eğer bu koşuda TÜM hedefli kategoriler mevcutsa VE hiç "_other" stem yoksa (n_unknown=0,
# yaygın durum), o kalan pay hiçbir örneğe gitmez — WeightedRandomSampler yalnız GÖRECELİ
# ağırlıklarla çalıştığından bu, hedeflerin kendi aralarında ~%(1-sum(target_share)) kadar
# ZARARSIZ bir şekilde yeniden ölçeklenmesine yol açar (v1'in sum=%40'ında bu etki yoktu çünkü
# hedefsiz "diğer" kategoriler — hair/complex/thin/general — her zaman mevcuttu). Bu yüzden
# tolerans 1e-6 yerine 2e-2 (v2-hazırlık raporu: "within 1%" + güvenlik payı).
_TARGET_SHARE_TOLERANCE = 1e-6 if SAMPLER_PRESET == "v1" else 2e-2
for cat, target in CATEGORY_TARGET_SHARE.items():
    if cat in achieved_shares:
        assert abs(achieved_shares[cat] - target) < _TARGET_SHARE_TOLERANCE, (
            f"{cat} için beklenen pay %{target*100:.1f} ama hesaplanan %{achieved_shares[cat]*100:.1f} "
            f"— train_colab_lib.compute_sample_weights mantığı kontrol edilmeli."
        )

_epoch_num_samples = tcl.resolve_sampler_num_samples(len(train_dataset), EPOCH_NUM_SAMPLES)
print(f"Epoch başına örnek sayısı: {_epoch_num_samples} "
      f"(EPOCH_NUM_SAMPLES={EPOCH_NUM_SAMPLES!r}, dataset büyüklüğü={len(train_dataset)}).")
sampler = WeightedRandomSampler(sample_weights, num_samples=_epoch_num_samples, replacement=True)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH,
    sampler=sampler,
    shuffle=False,  # sampler verildiğinde shuffle=True ile birlikte KULLANILAMAZ (torch kısıtı) — sampler zaten karıştırıyor.
    num_workers=NUM_WORKERS,  # BİLİNÇLİ SAPMA: resmi formül min(num_workers, batch_size) bs=2'de 2 işçiye
                              # düşerdi — ~13.7k iterasyonluk epoch'ta veri okuma darboğaz olur; A100
                              # VM'lerinin bol CPU'su varken işçi sayısını batch'ten ayrıştırıyoruz.
    pin_memory=True,
    drop_last=True,  # resmi train.py ile AYNI (son eksik batch atlanır).
)
print(f"train_loader hazır: {len(train_loader)} batch/epoch (batch_size={BATCH}, accum={ACCUM} -> efektif batch={BATCH * ACCUM}).")

## (f) Kayıp Fonksiyonu, Optimizer, Hızlı-Değerlendirme

`PixLoss`/`ClsLoss` resmi `loss.py`'den DEĞİŞTİRİLMEDEN kullanılıyor.
`effective_lr` (LR=None ise), resmi formülü (`config.py`) efektif batch'e
(`BATCH*ACCUM`) göre uyarlıyor (bkz. `train_colab_lib.effective_lr` docstring'i
— gradyan biriktirme resmi kodda olmadığı için bu uyarlama BİZİM eklentimiz).

`run_quick_eval`, `bgr/segmenter.py`'nin ZATEN kullandığı çıkarım desenini
(`model(inp)[-1].sigmoid()`, `BiRefNet.forward`'ın `self.training=False`'ta
döndürdüğü yapı) ve projenin ortak metrik modülünü (`benchmark.metrics.mae`)
kullanıyor — VAL_HOLDOUT'tan sabit `EVAL_STEMS` üzerinde, eğitim çözünürlüğünde
(1024×1024) yaklaşık bir MAE. **Bu, 155 görsellik tam benchmark'ın YERİNE
GEÇMEZ** — yalnız epoch'lar arası hızlı bir trend sinyali; tam ölçüm eğitim
bittikten sonra lokalde (`benchmark/run.py`) yapılacak.

In [ ]:
from loss import ClsLoss, PixLoss  # noqa: E402
from torchvision import transforms

pix_loss = PixLoss()
cls_loss = ClsLoss()
criterion_gdt = torch.nn.BCELoss() if config.out_ref else None
BASE_LAMBDAS_PIX_LAST = dict(pix_loss.lambdas_pix_last)  # finetune-reweight'in HER epoch BU tabandan yeniden hesaplanması için (bkz. eğitim döngüsü hücresi notu)

lr = tcl.effective_lr(config.task, BATCH, ACCUM, base_lr_override=LR)
if LR is None:  # yalnız resmi FORMÜL kullanıldığında ölçeklenir — elle girilen LR AYNEN kullanılır.
    lr *= LR_SCALE
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-2)
lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[m if m > 0 else EPOCHS + m + 1 for m in config.lr_decay_epochs],
    gamma=config.lr_decay_rate,
)
if payload is not None and "optimizer" in payload:
    optimizer.load_state_dict(payload["optimizer"])
    lr_scheduler.load_state_dict(payload["lr_scheduler"])
    print("Optimizer + lr_scheduler durumu checkpoint'ten geri yüklendi.")
print(f"Öğrenme oranı: {lr:.3e} (LR parametresi={'otomatik' if LR is None else LR}, "
      f"LR_SCALE={LR_SCALE if LR is None else 'uygulanmadı (LR elle set edildi)'})")

_quick_eval_transform = transforms.Compose([
    transforms.Resize(config.size[::-1]),  # config.size = (wid, hei); Resize (hei, wid) bekler.
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


@torch.no_grad()
def run_quick_eval(model, eval_stems, im_dir, gt_dir, device) -> float:
    import numpy as np
    from PIL import Image

    model.eval()
    maes = []
    for stem in eval_stems:
        img_path, gt_path = im_dir / f"{stem}.jpg", gt_dir / f"{stem}.png"
        if not (img_path.exists() and gt_path.exists()):
            continue
        img = Image.open(img_path).convert("RGB")
        gt = Image.open(gt_path).convert("L").resize(config.size, Image.BILINEAR)
        inp = _quick_eval_transform(img).unsqueeze(0).to(device)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            pred = model(inp)[-1].sigmoid()
        pred_np = pred.float().cpu().squeeze().numpy()
        gt_np = np.asarray(gt, dtype=np.float32) / 255.0
        maes.append(bm.mae(pred_np, gt_np))
    model.train()
    return float(sum(maes) / len(maes)) if maes else float("nan")


print(f"Hazır: {len(EVAL_STEMS)} sabit VAL_HOLDOUT görseliyle hızlı-değerlendirme fonksiyonu tanımlandı.")

## (g) Eğitim Döngüsü — Checkpoint/Resume + Log + Periyodik Hızlı-Değerlendirme

Her epoch sonunda:
1. `MyDrive/bg-remover-status/train_log.txt`'e bir satır eklenir (`epoch, loss,
   lr, time` — kontrolcü bu dosyayı DIŞARIDAN izliyor) + ölçülmüş epoch süresi ve
   yaklaşık Colab birimi maliyeti konsola yazdırılır (ilk epoch'tan itibaren —
   maliyet tablosundaki teorik tahmini gerçek ölçümle karşılaştırın).
2. Checkpoint (model+optimizer+lr_scheduler+epoch) ÖNCE yerel diske, SONRA Drive'a
   (`bg-remover-checkpoints/epoch_<N>.pth`) yazılır; her iki yerde de yalnız
   son `KEEP_LAST_N_CHECKPOINTS` tutulur (`train_colab_lib.prune_old_checkpoints`).
3. `N_EVAL_EVERY` epoch'ta bir (+ son epoch'ta HER ZAMAN) hızlı-değerlendirme
   MAE'si hesaplanıp log'a eklenir.

**Drive dayanıklılığı:** Drive-mount'a yazmak (checkpoint kopyası, log satırı)
ara sıra geçici I/O hatası verebilir (kota, senkronizasyon takılması). Bunlar
eğitimi ÖLDÜRMEZ: yerel checkpoint zaten kaydedilmiş olur, Drive kopyası/log
satırı try/except ile sarılıdır — uyarı basılır, bir SONRAKİ epoch'ta yeniden
denenir (checkpoint zaten her epoch kopyalandığından kayıp en fazla 1 epoch'luk
Drive gecikmesidir). FATAL traceback yazımı da best-effort'tur.

**Drive kotası:** her checkpoint paketi (model fp32 + AdamW momentleri ×2 +
scheduler) ~2.5-3GB; `KEEP_LAST_N_CHECKPOINTS=3` ile Drive'da **~8-9GB** yer
tutar. Drive'ınızda bu kadar boş alan olduğundan emin olun (15GB'lık ücretsiz
kotanın büyük kısmı budur — gerekirse parametreyi 2'ye düşürün).

**torch.compile notu:** `COMPILE_MODEL=True` ise `model.eval()` ↔ `model.train()`
geçişi (hızlı-değerlendirme hücresi) İLK SEFERİNDE bir kez daha derleme tetikler
(eval grafiği trainden farklı) — ilk hızlı-değerlendirmenin birkaç dakika uzun
sürmesi normaldir, sonrakiler önbellekten çalışır.

**Fine-tune hilesi (resmi `train.py::train_epoch`, Matting dalı) — resume-güvenli
hale getirildi:** resmi kod, son `|finetune_last_epochs|` (Matting: 10) epoch'ta
`ssim`/`mse` ağırlıklarını HER epoch KÜMÜLATİF olarak ×0.9 çarpıyor — ama bu
mantık, `PixLoss()`'un SÜREÇ BOYUNCA canlı kaldığı, HİÇ yeniden başlamayan tek
bir çalıştırma varsayımına dayanıyor. Bizim çok-oturumlu (birkaç saatte bir
Colab kopması) senaryomuzda `PixLoss()` her oturumda SIFIRDAN kuruluyor —
kümülatif çarpımı olduğu gibi taklit etmek YANLIŞ olurdu (her resume'de
sayaç sıfırlanır, üstel decay bozulur). Bunun yerine HER epoch'ta, tabandan
(`BASE_LAMBDAS_PIX_LAST`) `0.9 ** n` ile MUTLAK epoch numarasına göre yeniden
hesaplıyoruz — kesintisiz çalışan resmi koduyla MATEMATİKSEL OLARAK ÖZDEŞ
sonuç verir, ama resume'e karşı sağlam.

In [ ]:
import time
import traceback

STATUS_DIR = Path(DRIVE_ROOT) / DRIVE_STATUS_SUBDIR
TRAIN_LOG_PATH = STATUS_DIR / "train_log.txt"
STATUS_DIR.mkdir(parents=True, exist_ok=True)

UNITS_PER_HOUR_A100 = 13  # yaklaşık (Colab A100 ~11-13 birim/saat); kesin değeri Colab'ın "Kaynaklar" panelinden doğrulayın.


def log_epoch_row(epoch: int, loss: float, lr_now: float, elapsed_sec: float, eval_mae: float | None) -> None:
    row = f"epoch={epoch}\tloss={loss:.6f}\tlr={lr_now:.8f}\ttime_sec={elapsed_sec:.1f}"
    if eval_mae is not None:
        row += f"\teval_mae={eval_mae:.6f}"
    print(row)
    # Drive'a log yazımı best-effort: geçici bir Drive I/O hatası eğitimi ÖLDÜRMEMELİ
    # (satır konsola zaten basıldı; bir sonraki epoch'un satırı yine denenecek).
    try:
        with open(TRAIN_LOG_PATH, "a") as f:
            f.write(row + "\n")
    except OSError as e:
        print(f"  UYARI: train_log.txt'e yazılamadı ({e}) — eğitim devam ediyor, sonraki epoch'ta tekrar denenecek.")


def save_and_sync_checkpoint(epoch: int) -> None:
    raw_state = model.state_dict()  # torch.compile ise '_orig_mod.' önekli olabilir -- resmi train.py ile AYNI davranış
                                     # (öneki KALDIRMADAN kaydeder); yükleme sırasında her zaman check_state_dict uygulanır.
    payload_out = {
        "model": raw_state,
        "optimizer": optimizer.state_dict(),
        "lr_scheduler": lr_scheduler.state_dict(),
        "epoch": epoch,
    }
    # 1) ÖNCE yerel disk — bu her zaman başarılı olmalı (başarısızsa gerçek bir sorun var, hata yükselir).
    local_path = local_ckpt_dir / f"epoch_{epoch}.pth"
    torch.save(payload_out, local_path)
    tcl.prune_old_checkpoints(local_ckpt_dir, KEEP_LAST_N_CHECKPOINTS)

    # 2) SONRA Drive — best-effort: geçici Drive I/O hatası (kota/senkron takılması)
    # eğitimi öldürmez; yerel kopya güvende, bir SONRAKİ epoch yeni bir Drive
    # kopyası deneyecek (en kötü durumda Drive 1 epoch geriden gelir).
    try:
        drive_path = drive_ckpt_dir / f"epoch_{epoch}.pth"
        shutil.copy2(local_path, drive_path)
        tcl.prune_old_checkpoints(drive_ckpt_dir, KEEP_LAST_N_CHECKPOINTS)
        print(f"  checkpoint kaydedildi + Drive'a kopyalandı: {drive_path}")
    except OSError as e:
        print(f"  UYARI: checkpoint Drive'a kopyalanamadı ({e}) — YEREL kopya güvende: {local_path}; "
              f"sonraki epoch'ta yeniden denenecek.")


def train_one_epoch(epoch: int) -> float:
    model.train()

    # --- Fine-tune hilesi: mutlak epoch numarasına göre TABANDAN yeniden hesapla (resume-güvenli, bkz. yukarıdaki not) ---
    pix_loss.lambdas_pix_last = dict(BASE_LAMBDAS_PIX_LAST)
    if tcl.should_apply_finetune_reweight(epoch, EPOCHS, config.finetune_last_epochs):
        n = epoch - (EPOCHS + config.finetune_last_epochs)
        if config.task == "Matting":
            pix_loss.lambdas_pix_last["mse"] = BASE_LAMBDAS_PIX_LAST["mse"] * (0.9 ** n)
            pix_loss.lambdas_pix_last["ssim"] = BASE_LAMBDAS_PIX_LAST["ssim"] * (0.9 ** n)
        else:
            pix_loss.lambdas_pix_last["bce"] = BASE_LAMBDAS_PIX_LAST["bce"] * 0
            pix_loss.lambdas_pix_last["iou"] = BASE_LAMBDAS_PIX_LAST["iou"] * (0.5 ** n)
            pix_loss.lambdas_pix_last["mae"] = BASE_LAMBDAS_PIX_LAST["mae"] * (0.9 ** n)

    running_sum, running_n = 0.0, 0
    n_batches = len(train_loader)
    optimizer.zero_grad()
    for micro_step, batch in enumerate(train_loader):
        inputs = batch[0].to(device, non_blocking=True)
        gts = batch[1].to(device, non_blocking=True)
        class_labels = batch[2].to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            scaled_preds, class_preds_lst = model(inputs)
            if config.out_ref:
                (outs_gdt_pred, outs_gdt_label), scaled_preds = scaled_preds

        # neden: BCE (BCELoss / binary_cross_entropy) autocast altında yasak; resmi BiRefNet
        # train.py/loss.py ile AYNI matematik, sadece bu blok (gdt + pix_loss) fp32’de hesaplanıyor.
        with torch.autocast(device_type="cuda", enabled=False):
            if config.out_ref:
                loss_gdt = None
                for gi, (gp, gl) in enumerate(zip(outs_gdt_pred, outs_gdt_label)):
                    gp_i = torch.nn.functional.interpolate(gp.float(), size=gl.shape[2:], mode="bilinear", align_corners=True).sigmoid()
                    gl_i = gl.float().sigmoid()
                    li = criterion_gdt(gp_i, gl_i)
                    loss_gdt = li if gi == 0 else loss_gdt + li
            loss_cls = 0.0 if None in class_preds_lst else cls_loss(class_preds_lst, class_labels)
            # pix_loss (resmi PixLoss) da ’bce’ bileşeninde ham BCELoss kullanıyor -- aynı nedenle fp32’ye yükseltiliyor.
            scaled_preds_f = [sp.float() for sp in scaled_preds]
            loss_pix, _loss_dict_pix = pix_loss(scaled_preds_f, torch.clamp(gts, 0, 1).float(), pix_loss_lambda=1.0)
            loss = loss_pix + loss_cls
            if config.out_ref:
                loss = loss + loss_gdt * 1.0

        (loss / ACCUM).backward()
        if (micro_step + 1) % ACCUM == 0 or (micro_step + 1) == n_batches:
            optimizer.step()
            optimizer.zero_grad()

        running_sum += loss.item() * inputs.size(0)
        running_n += inputs.size(0)
        if micro_step % 200 == 0:
            print(f"  epoch {epoch} iter {micro_step}/{n_batches} loss={loss.item():.5g}")

    lr_scheduler.step()
    return running_sum / max(running_n, 1)


def main() -> None:
    for epoch in range(epoch_st, EPOCHS + 1):
        t0 = time.time()
        avg_loss = train_one_epoch(epoch)
        elapsed = time.time() - t0
        current_lr = optimizer.param_groups[0]["lr"]

        eval_mae = None
        if epoch % N_EVAL_EVERY == 0 or epoch == EPOCHS:
            eval_mae = run_quick_eval(model, EVAL_STEMS, local_val_im, local_val_gt, device)

        log_epoch_row(epoch, avg_loss, current_lr, elapsed, eval_mae)
        save_and_sync_checkpoint(epoch)

        # ÖLÇÜLMÜŞ maliyet raporu (parametre hücresindeki teorik tabloyla karşılaştırın):
        hours = elapsed / 3600
        est_units = hours * UNITS_PER_HOUR_A100
        remaining = EPOCHS - epoch
        print(f"  MALİYET: bu epoch {hours:.2f} saat ≈ {est_units:.0f} birim "
              f"(A100 ~{UNITS_PER_HOUR_A100} birim/saat varsayımıyla); "
              f"kalan {remaining} epoch ≈ {remaining * hours:.1f} saat ≈ {remaining * est_units:.0f} birim. "
              f"Bütçenizi aşacaksa şimdi durdurun — RESUME='auto' kaldığı yerden devam eder.")

    print("EĞİTİM TAMAMLANDI.")


try:
    main()
except Exception:
    tb = traceback.format_exc()
    print(tb)
    try:  # FATAL kaydı da best-effort — Drive yazılamıyorsa asıl hatayı gölgelemesin.
        with open(TRAIN_LOG_PATH, "a") as f:
            f.write(f"epoch=FATAL\ttraceback={tb!r}\n")
    except OSError as log_err:
        print(f"UYARI: FATAL kaydı train_log.txt'e yazılamadı ({log_err}).")
    raise

## Eğitim Sonrası

- **Tam benchmark (155 görsel) LOKALDE (M4) yapılır** — bu notebook'un işi
  yalnız fine-tune + hızlı MAE trendi. Eğitim bitince Drive'daki son
  checkpoint'i (`bg-remover-checkpoints/epoch_<EPOCHS>.pth`) indirip
  `scripts/export_birefnet.py`/`benchmark/run.py` akışıyla lokalde
  değerlendirin (bkz. `docs/reports/2026-07-faz2-veri.md` §5 hedefleri:
  camouflage MAE ≤0.075, transparent MAE <0.06).
- `train_log.txt`'i (`MyDrive/bg-remover-status/`) kontrolcü zaten dışarıdan
  izliyor; bu notebook'u tekrar çalıştırmak (`RESUME="auto"`) kaldığı yerden
  devam eder.
- Faz 4 (paketleme/ONNX export) bu checkpoint'i girdi olarak kullanacak.